## Mujoco Jacobian
- Get Jacobian of the specific body
- Get pseudo-inverse of Jacobian
    - Damped Least Squares
    - Singular Value Decomposition

### Singular Value Decomposition
- Decompose jacobian into S, V and sigma
- V: direction of motion in joint space, U: corresponding motion direction in end-effector space
- Sigma: Contains singular values (degree of amplification)
    - large singular value -> small joint move makes large 
    - Thresholding sigma to avoid singular value
- $J = U \Sigma V^T$
- $J^+ = V \Sigma^+ U^T$
</br>

### Damped Least Squares
- pseudo-inverse of Jacobian
- Damping factor: reduce singularity issue
    - lambda damping
    - e with unit error vector
- $\Delta\theta = J^T(JJ^T + \lambda^2I)^{-1} \vec{e}$

Create UR environment

In [1]:
import mujoco
import mujoco_viewer # new viewer
import numpy as np
import time

In [2]:
model_path = "../ur5e_mjcf/scene.xml"

# declare model & data
model = mujoco.MjModel.from_xml_path(model_path)
data = mujoco.MjData(model)

In [3]:
def get_body_name (model, data):
    body_names = [mujoco.mj_id2name(model,mujoco.mjtObj.mjOBJ_BODY, body_idx) for body_idx in range(model.nbody)]
    return body_names

body_names = get_body_name(model, data)

print(body_names)

['world', 'base', 'shoulder_link', 'upper_arm_link', 'forearm_link', 'wrist_1_link', 'wrist_2_link', 'wrist_3_link']


Get Mujoco Jacobian

In [4]:
""" GET MUJOCO JACOBIAN """

body_name = "wrist_3_link"

# initialize positional & rotational jacobian
Jacobian_p = np.zeros((3,model.nu))
Jacobian_r = np.zeros((3,model.nu))
# get jacobian of end-effector

mujoco.mj_resetData(model, data)
mujoco.mj_forward(model, data)

mujoco.mj_jacBody(model, data, Jacobian_p, Jacobian_r, data.body(body_name).id)
print(Jacobian_p) 

# shape: 3x6 because 6 dof arm

"""
IMPORTANT: jacobian is not calculated if no step / forward 
"""
# mujoco.mj_jac(model, data, p, r, )

[[-8.1700000e-01 -4.4408921e-17 -4.4408921e-17 -4.4408921e-17
   0.0000000e+00  0.0000000e+00]
 [-1.3400000e-01 -1.0000000e-01 -1.0000000e-01 -1.0000000e-01
   0.0000000e+00  0.0000000e+00]
 [ 0.0000000e+00 -8.1700000e-01 -3.9200000e-01  0.0000000e+00
   0.0000000e+00  0.0000000e+00]]


'\nIMPORTANT: jacobian is not calculated if no step / forward \n'

### Calculate delta theta with inverse jacobian

Singular Value Decomposition

In [5]:
U, Sigma, V = np.linalg.svd(Jacobian_p, compute_uv=True)

In [6]:
print(f"U, S, V from SVD:\nU: {U}\nS (vector): {Sigma}\nV: {V}")

U, S, V from SVD:
U: [[-0.10311419  0.98124184  0.1628862 ]
 [-0.16373586  0.14478032 -0.97582233]
 [-0.98110042 -0.12729145  0.14573557]]
S (vector): [0.91724043 0.82682662 0.10782269]
V: [[ 1.15765614e-01  8.91731985e-01  4.37142692e-01  1.78509208e-02
   0.00000000e+00  0.00000000e+00]
 [-9.93043913e-01  1.08268259e-01  4.28387451e-02 -1.75103597e-02
   0.00000000e+00  0.00000000e+00]
 [-2.14967301e-02 -1.99250545e-01  3.75189016e-01  9.05025035e-01
   0.00000000e+00  0.00000000e+00]
 [-1.15056905e-32  3.91651976e-01 -8.16274654e-01  4.24622678e-01
   0.00000000e+00  0.00000000e+00]
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00
   1.00000000e+00  0.00000000e+00]
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00
   0.00000000e+00  1.00000000e+00]]


In [7]:
# Get inverse Sigma by dividing values
upper_bound = 0.05 # change value 
Sigma_rev = 1/Sigma
Sigma_clipped_rev = np.minimum(1 / Sigma, upper_bound)

"""
NOTICE
- upper bound must be small
    - (2.5 -> angle update ~0.9)
    - 0.2 -> dq within 0.15
    - 0.05 -> dq within 0.04
"""

'\nNOTICE\n- upper bound must be small\n    - (2.5 -> angle update ~0.9)\n    - 0.2 -> dq within 0.15\n    - 0.05 -> dq within 0.04\n'

In [10]:
sigma_threshold = 0.2

Sigma_clipped_rev = np.zeros_like(Sigma)
for i, value in enumerate(Sigma):
    if Sigma[i] < sigma_threshold:
        Sigma_clipped_rev[i] = 0
    else:
        Sigma_clipped_rev[i] = 1/Sigma[i]

In [11]:
Sigma_clipped_rev

array([1.09022669, 1.2094434 , 0.        ])

In [12]:
# inverse matrix of Sigma: size should be 6x3
S_rev_matrix = np.zeros((6,3))
for i, value in enumerate(Sigma_clipped_rev):
    S_rev_matrix[i,i] = value

In [13]:
S_rev_matrix

array([[1.09022669, 0.        , 0.        ],
       [0.        , 1.2094434 , 0.        ],
       [0.        , 0.        , 0.        ],
       [0.        , 0.        , 0.        ],
       [0.        , 0.        , 0.        ],
       [0.        , 0.        , 0.        ]])

In [14]:
print(f"shape U, V, S: {U.shape}, {V.shape}, {S_rev_matrix}")

shape U, V, S: (3, 3), (6, 6), [[1.09022669 0.         0.        ]
 [0.         1.2094434  0.        ]
 [0.         0.         0.        ]
 [0.         0.         0.        ]
 [0.         0.         0.        ]
 [0.         0.         0.        ]]


In [ ]:
J_inverse = V.T @ S_rev_matrix @ U.T

In [16]:
J_inverse

array([[ 1.04525458,  0.13548025, -0.26110917],
       [ 0.24012391,  0.19622564,  1.04551339],
       [-0.23404526, -0.03105212,  0.05366835],
       [ 0.46479551,  0.06857967, -0.06029553],
       [ 0.        ,  0.        ,  0.        ],
       [ 0.        ,  0.        ,  0.        ]])

In [17]:
data.body(body_name).xpos
# data.body(body_name).xquat

array([-0.134,  0.817,  0.063])

In [18]:
# delta joint = Jacobian inverse * unit error vector

goal_pos = [0.5, 0.5, 0.5]
error = goal_pos - data.body(body_name).xpos

# make it unit vector
error /= np.linalg.norm(error)
error

array([ 0.76136508, -0.38068254,  0.5247895 ])

In [19]:
dq = J_inverse @ error
dq

array([ 0.60721802,  0.65679673, -0.1382083 ,  0.29612953,  0.        ,
        0.        ])

In [20]:
qpos_before = data.qpos.copy()
data.qpos += dq
print(f"qpos before update: {qpos_before} \n qpos after update: {data.qpos}")

qpos before update: [0. 0. 0. 0. 0. 0.] 
 qpos after update: [ 0.60721802  0.65679673 -0.1382083   0.29612953  0.          0.        ]
